# Week 7 Implementation: ES Adaptation Under Environment Perturbation

**Date:** February 25, 2026  
**Course:** STAT 4830

This notebook studies transfer adaptation with Evolution Strategies (ES):
1. Pretrain a policy on baseline GridWorld.
2. Perturb the environment with Gaussian transition noise.
3. Continue training from the same pretrained checkpoint using:
   - **Full ES** (`param_mode='all'`)
   - **LoRA-only ES** (`param_mode='lora'`, pretrained base frozen)

The focus is adaptation speed, not just final performance.

## Problem Setup

### Adaptation Question

**Goal:** Measure how quickly each method re-learns a good policy after an environment shift.

**Methods compared from the same initialization:**
1. **Full ES adaptation** (`param_mode='all'`)
2. **LoRA-only ES adaptation** (`param_mode='lora'`)

**Environment shift:** Gaussian transition perturbation (action noise) at controlled levels.

**Fairness controls:**
- Same pretrained checkpoint for both methods
- Same perturbed map instance per seed
- Same ES hyperparameters and evaluation schedule

### Mathematical Formulation

We optimize expected return:
$$
\max_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[\sum_{t=0}^{T} r_t\right]
$$

ES gradient estimator:
$$
\nabla_\theta J(\theta) \approx \frac{1}{N\sigma}\sum_{i=1}^{N} F_i\, s(\epsilon_i)
$$
where $s(\epsilon)$ is the score-function term for the selected noise distribution.

For rank-1 LoRA on each linear layer:
$$
W' = W + \alpha\, b a^\top
$$
with vector factors $a$ and $b$ (rank = 1). In LoRA-only mode, ES updates only adapter vectors while $W$ is frozen.

### Data Requirements

**Environment:**
- Grid size: 8x8
- Obstacles: 8
- State space: one-hot position (64 dims)
- Action space: 4 actions
- Max steps per episode: 50

**Reward scheme:**
- **Training:** distance-based shaped rewards (dense signal)
- **Evaluation:** sparse reward GridWorld

**Perturbation protocol:**
- Gaussian transition noise std levels: `0.00`, `0.10`, `0.20`, `0.30`
- Same world seed pairings for both methods at each level
- Evaluate adaptation over a fixed post-perturbation ES budget

### Success Metrics

**Primary adaptation metrics:**
1. **Time-to-threshold**: first eval point where success reaches `0.6`, `0.8`, `0.9`
2. **AUC(success vs iteration)** over adaptation horizon
3. **Threshold interactions**: cumulative environment interactions needed to hit threshold

**Secondary:**
4. Final `eval_success`, `eval_reward`, and `eval_steps`

**Diagnostics:**
5. `gradient_norm`, `avg_fitness`, and `fitness_std` trends

## Implementation

In [ ]:
# All required imports
import sys
sys.path.append('../src')

import time
from copy import deepcopy

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from model import GridWorld, PolicyNetwork
from utils import train_es, evaluate_policy

# Reproducibility
GLOBAL_SEED = 42
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 4)

print('Imports successful!')
print(f'PyTorch version: {torch.__version__}')
print(f'Device: {device}')

Imports successful!
PyTorch version: 2.10.0+cpu
Device: cpu
Noise types: ('gaussian', 'cauchy', 'laplace')


### Environment Implementation

In [ ]:
# Distance-based reward shaping (same strategy as Week 4)
class ShapedRewardEnvComparison(GridWorld):
    """GridWorld with distance-based reward shaping."""

    def reset(self):
        state = super().reset()
        self.prev_dist = abs(self.agent_pos[0] - self.goal_pos[0]) + abs(self.agent_pos[1] - self.goal_pos[1])
        return state

    def step(self, action):
        state, reward, done, info = super().step(action)
        curr_dist = abs(self.agent_pos[0] - self.goal_pos[0]) + abs(self.agent_pos[1] - self.goal_pos[1])
        shaped_reward = reward + 0.2 * (self.prev_dist - curr_dist) - 0.01
        self.prev_dist = curr_dist
        return state, shaped_reward, done, info


class PerturbedTransitionGridWorld(GridWorld):
    """GridWorld with Gaussian transition perturbation via action slips."""

    def __init__(self, transition_noise_std: float = 0.0, **kwargs):
        super().__init__(**kwargs)
        self.transition_noise_std = float(transition_noise_std)

    def step(self, action):
        # Convert Gaussian noise into occasional action-slip perturbations.
        applied_action = int(action)
        if self.transition_noise_std > 0:
            slip_trigger = abs(self.rng.normal(0.0, self.transition_noise_std))
            if slip_trigger > 1.0:
                alternatives = [a for a in range(self.n_actions) if a != action]
                applied_action = int(self.rng.choice(alternatives))
        return super().step(applied_action)


class ShapedPerturbedTransitionGridWorld(PerturbedTransitionGridWorld):
    """Perturbed transition dynamics plus dense distance-based shaping."""

    def reset(self):
        state = super().reset()
        self.prev_dist = abs(self.agent_pos[0] - self.goal_pos[0]) + abs(self.agent_pos[1] - self.goal_pos[1])
        return state

    def step(self, action):
        state, reward, done, info = super().step(action)
        curr_dist = abs(self.agent_pos[0] - self.goal_pos[0]) + abs(self.agent_pos[1] - self.goal_pos[1])
        shaped_reward = reward + 0.2 * (self.prev_dist - curr_dist) - 0.01
        self.prev_dist = curr_dist
        return state, shaped_reward, done, info


def align_world_layout(target_env: GridWorld, source_env: GridWorld):
    """Copy obstacle/start/goal layout to ensure paired comparisons."""
    target_env.goal_pos = source_env.goal_pos
    target_env.start_pos = source_env.start_pos
    target_env.obstacles = set(source_env.obstacles)


# Global experiment controls
GRID_SIZE = 8
N_OBSTACLES = 8
MAX_STEPS = 50
PERTURB_LEVELS = [0.0, 0.10, 0.20, 0.30]
SEEDS = [11, 22, 33, 44, 55]

# Baseline example env (for quick sanity checks)
example_env = ShapedRewardEnvComparison(size=GRID_SIZE, n_obstacles=N_OBSTACLES, max_steps=MAX_STEPS, seed=123)
state = example_env.reset()

print(f'Base environment: {GRID_SIZE}x{GRID_SIZE}, obstacles={N_OBSTACLES}, max_steps={MAX_STEPS}')
print(f'State shape: {state.shape}, action space: {example_env.n_actions}')
print(f'Perturbation levels (Gaussian transition std): {PERTURB_LEVELS}')
print(f'Seeds: {SEEDS}')

example_env.render()

Training environment: 8x8 grid (shaped rewards)
State shape: (64,)
Action space: 4
Start: (7, 0) | Goal: (0, 7) | Obstacles: 8


<Figure size 800x800 with 1 Axes>

### Policy Setup: Standard vs Rank-1 LoRA

In [ ]:
state_dim = example_env._get_state().shape[0]
action_dim = example_env.n_actions

standard_policy = PolicyNetwork(
    state_dim=state_dim,
    action_dim=action_dim,
    hidden_dim=64,
    n_layers=2,
    use_lora=False
).to(device)

lora_policy = PolicyNetwork(
    state_dim=state_dim,
    action_dim=action_dim,
    hidden_dim=64,
    n_layers=2,
    use_lora=True,
    lora_rank=1,
    lora_alpha=1.0,
    lora_init_scale=1e-3
).to(device)

total_standard = sum(p.numel() for p in standard_policy.parameters())
total_lora_model = sum(p.numel() for p in lora_policy.parameters())
lora_trainable = lora_policy.count_lora_parameters()

print('Parameter counts:')
print(f'  Full ES searched params: {total_standard}')
print(f'  LoRA-only searched params: {lora_trainable}')
print(f'  Search-space compression: {total_standard / lora_trainable:.2f}x')

Standard ES policy parameters:
  Total parameters searched: 8580

LoRA policy parameters:
  Total model parameters: 8904
  LoRA search parameters (rank-1 only): 324
  Compression factor in ES search space: 27.48x


### ES Experiment Utilities

In [ ]:
PRETRAIN_CFG = {
    'N': 50,
    'sigma': 0.10,
    'alpha': 0.05,
    'n_iterations': 80,
    'n_eval_episodes': 5,
    'max_steps': MAX_STEPS,
    'eval_every': 5,
    'noise_type': 'gaussian'
}

ADAPT_CFG = {
    'N': 50,
    'sigma': 0.10,
    'alpha': 0.05,
    'n_iterations': 80,
    'n_eval_episodes': 5,
    'max_steps': MAX_STEPS,
    'eval_every': 5,
    'noise_type': 'gaussian'
}


def make_policy(use_lora: bool, lora_rank: int = 1):
    return PolicyNetwork(
        state_dim=state_dim,
        action_dim=action_dim,
        hidden_dim=64,
        n_layers=2,
        use_lora=use_lora,
        lora_rank=lora_rank,
        lora_alpha=1.0,
        lora_init_scale=1e-3
    ).to(device)


def copy_pretrained_into_lora_base(pretrained_full_policy, lora_policy):
    """Copy pretrained Linear weights into LoRA base layers."""
    src_linears = [m for m in pretrained_full_policy.network if isinstance(m, torch.nn.Linear)]
    dst_lora_wrappers = [m for m in lora_policy.network if hasattr(m, 'base')]

    if len(src_linears) != len(dst_lora_wrappers):
        raise ValueError('Architecture mismatch while loading pretrained weights into LoRA base.')

    for src, dst in zip(src_linears, dst_lora_wrappers):
        dst.base.weight.data.copy_(src.weight.data)
        if src.bias is not None and dst.base.bias is not None:
            dst.base.bias.data.copy_(src.bias.data)


def make_source_envs(seed: int):
    train_env = ShapedRewardEnvComparison(
        size=GRID_SIZE,
        n_obstacles=N_OBSTACLES,
        max_steps=MAX_STEPS,
        seed=seed
    )
    eval_env = GridWorld(
        size=GRID_SIZE,
        n_obstacles=N_OBSTACLES,
        max_steps=MAX_STEPS,
        seed=seed
    )
    align_world_layout(eval_env, train_env)
    return train_env, eval_env


def make_target_envs(seed: int, transition_noise_std: float, source_layout_env: GridWorld):
    train_env = ShapedPerturbedTransitionGridWorld(
        size=GRID_SIZE,
        n_obstacles=N_OBSTACLES,
        max_steps=MAX_STEPS,
        seed=seed,
        transition_noise_std=transition_noise_std
    )
    eval_env = PerturbedTransitionGridWorld(
        size=GRID_SIZE,
        n_obstacles=N_OBSTACLES,
        max_steps=MAX_STEPS,
        seed=seed,
        transition_noise_std=transition_noise_std
    )
    align_world_layout(train_env, source_layout_env)
    align_world_layout(eval_env, source_layout_env)
    return train_env, eval_env


def pretrain_source_policy(seed: int):
    train_env, eval_env = make_source_envs(seed)
    pretrained_policy = make_policy(use_lora=False)

    _ = train_es(
        policy=pretrained_policy,
        env=train_env,
        N=PRETRAIN_CFG['N'],
        sigma=PRETRAIN_CFG['sigma'],
        alpha=PRETRAIN_CFG['alpha'],
        n_iterations=PRETRAIN_CFG['n_iterations'],
        n_eval_episodes=PRETRAIN_CFG['n_eval_episodes'],
        max_steps=PRETRAIN_CFG['max_steps'],
        eval_every=PRETRAIN_CFG['eval_every'],
        verbosity=0,
        noise_type=PRETRAIN_CFG['noise_type'],
        param_mode='all'
    )

    src_reward, src_success, src_steps = evaluate_policy(
        policy=pretrained_policy,
        env=eval_env,
        n_episodes=30,
        max_steps=MAX_STEPS,
        deterministic=True
    )

    return pretrained_policy, train_env, {
        'pretrain_reward': float(src_reward),
        'pretrain_success': float(src_success),
        'pretrain_steps': float(src_steps)
    }


def first_threshold_hit(history, threshold: float):
    for idx, value in enumerate(history['eval_success']):
        if value >= threshold:
            return int(history['iteration'][idx]), float(history['cumulative_env_interactions'][idx])
    return None, None


def adaptation_auc(history):
    x = np.asarray(history['iteration'], dtype=np.float64)
    y = np.asarray(history['eval_success'], dtype=np.float64)
    if len(x) < 2:
        return float('nan')
    return float(np.trapz(y, x) / (x[-1] - x[0]))


def run_adaptation(seed: int, perturb_std: float, method: str):
    pretrained_policy, source_layout_env, pretrain_metrics = pretrain_source_policy(seed)
    target_train_env, target_eval_env = make_target_envs(seed + 100, perturb_std, source_layout_env)

    if method == 'full':
        policy = deepcopy(pretrained_policy).to(device)
        param_mode = 'all'
    elif method == 'lora':
        policy = make_policy(use_lora=True, lora_rank=1)
        copy_pretrained_into_lora_base(pretrained_policy, policy)
        param_mode = 'lora'
    else:
        raise ValueError("method must be 'full' or 'lora'")

    t0 = time.time()
    history = train_es(
        policy=policy,
        env=target_train_env,
        N=ADAPT_CFG['N'],
        sigma=ADAPT_CFG['sigma'],
        alpha=ADAPT_CFG['alpha'],
        n_iterations=ADAPT_CFG['n_iterations'],
        n_eval_episodes=ADAPT_CFG['n_eval_episodes'],
        max_steps=ADAPT_CFG['max_steps'],
        eval_every=ADAPT_CFG['eval_every'],
        verbosity=0,
        noise_type=ADAPT_CFG['noise_type'],
        param_mode=param_mode
    )
    elapsed = time.time() - t0

    final_reward, final_success, final_steps = evaluate_policy(
        policy=policy,
        env=target_eval_env,
        n_episodes=40,
        max_steps=MAX_STEPS,
        deterministic=True
    )

    row = {
        'seed': int(seed),
        'method': method,
        'perturb_std': float(perturb_std),
        'runtime_sec': float(elapsed),
        'final_eval_reward': float(final_reward),
        'final_eval_success': float(final_success),
        'final_eval_steps': float(final_steps),
        'final_grad_norm': float(history['gradient_norm'][-1]),
        'final_fitness_std': float(history['fitness_std'][-1]),
        'auc_success': adaptation_auc(history),
        **pretrain_metrics
    }

    for threshold in (0.6, 0.8, 0.9):
        t_iter, t_interactions = first_threshold_hit(history, threshold)
        row[f'time_to_{threshold:.1f}'] = np.nan if t_iter is None else t_iter
        row[f'interactions_to_{threshold:.1f}'] = np.nan if t_interactions is None else t_interactions

    return row, history


print('Adaptation utilities ready.')

Utility functions ready.


### Run Adaptation Experiment

For each seed and perturbation level:
1. Pretrain a source policy on baseline GridWorld.
2. Initialize **both** adaptation variants from that source checkpoint.
3. Adapt on perturbed GridWorld using the same ES budget.

This cell produces:
- `results_df`: one row per run (seed, perturbation, method)
- `history_store`: trajectories for plotting adaptation dynamics

In [ ]:
rows = []
history_store = {}

device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f'Training device: {device} ({device_name})')
print(f'Running {len(SEEDS) * len(PERTURB_LEVELS) * 2} total adaptation runs...')

for perturb_std in PERTURB_LEVELS:
    for seed in SEEDS:
        for method in ('full', 'lora'):
            row, history = run_adaptation(seed=seed, perturb_std=perturb_std, method=method)
            rows.append(row)
            history_store[(seed, perturb_std, method)] = history

results_df = pd.DataFrame(rows)

print('Completed all runs.')
print('\nPer-method summary (mean across seeds):')
summary_cols = [
    'final_eval_success', 'final_eval_reward', 'final_eval_steps',
    'auc_success', 'time_to_0.8', 'interactions_to_0.8'
]
print(results_df.groupby(['perturb_std', 'method'])[summary_cols].mean().round(3))

Training device: cpu (CPU)
Running LoRA ES with gaussian noise and rank 1...
Iter    0/ 120 | Fitness: -0.233 | Eval Reward: -0.300 | Success: 0.00% | Grad Norm: 25.2947
Iter   20/ 120 | Fitness:  0.172 | Eval Reward:  0.700 | Success: 0.00% | Grad Norm: 23.6344
Iter   40/ 120 | Fitness:  2.878 | Eval Reward:  0.900 | Success: 0.00% | Grad Norm: 24.7266
Iter   60/ 120 | Fitness:  3.188 | Eval Reward:  2.100 | Success: 0.00% | Grad Norm: 26.1075
Iter   80/ 120 | Fitness:  3.440 | Eval Reward:  3.660 | Success: 100.00% | Grad Norm: 24.9893
Iter  100/ 120 | Fitness:  3.654 | Eval Reward:  3.660 | Success: 100.00% | Grad Norm: 24.1274
Iter  119/ 120 | Fitness:  3.565 | Eval Reward:  3.660 | Success: 100.00% | Grad Norm: 26.4641
Running LoRA ES with gaussian noise and rank 2...
Iter    0/ 120 | Fitness: -0.302 | Eval Reward: -0.500 | Success: 0.00% | Grad Norm: 25.8563
Iter   20/ 120 | Fitness:  1.405 | Eval Reward: -5.200 | Success: 0.00% | Grad Norm: 24.0088
Iter   40/ 120 | Fitness:  3.1

### Adaptation Analysis and Plots

In [ ]:
def _history_frame(metric_name: str):
    rows = []
    for (seed, perturb_std, method), hist in history_store.items():
        for iteration, value in zip(hist['iteration'], hist[metric_name]):
            rows.append({
                'seed': seed,
                'perturb_std': perturb_std,
                'method': method,
                'iteration': iteration,
                metric_name: value
            })
    return pd.DataFrame(rows)


curve_df = _history_frame('eval_success')
steps_curve_df = _history_frame('eval_steps')
grad_curve_df = _history_frame('gradient_norm')
fitstd_curve_df = _history_frame('fitness_std')

# 1) Adaptation learning curves (eval success)
fig, axes = plt.subplots(1, len(PERTURB_LEVELS), figsize=(4 * len(PERTURB_LEVELS), 4), sharey=True)
if len(PERTURB_LEVELS) == 1:
    axes = [axes]

for i, perturb_std in enumerate(PERTURB_LEVELS):
    ax = axes[i]
    subset = curve_df[curve_df['perturb_std'] == perturb_std]
    sns.lineplot(
        data=subset,
        x='iteration',
        y='eval_success',
        hue='method',
        estimator='mean',
        errorbar=('ci', 95),
        ax=ax
    )
    ax.set_title(f'Transition noise std = {perturb_std:.2f}')
    ax.set_ylim(0.0, 1.02)
    ax.grid(True, alpha=0.3)
    if i > 0:
        ax.get_legend().remove()

axes[0].set_ylabel('Eval success rate')
axes[-1].legend(loc='lower right')
plt.tight_layout()
plt.show()

# 2) Threshold crossing plot (iterations to 0.8 success)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.pointplot(
    data=results_df,
    x='perturb_std',
    y='time_to_0.8',
    hue='method',
    errorbar=('ci', 95),
    dodge=0.25,
    ax=axes[0]
)
axes[0].set_title('Iterations to 80% success')
axes[0].set_xlabel('Transition noise std')
axes[0].set_ylabel('Iteration')
axes[0].grid(True, alpha=0.3)

sns.pointplot(
    data=results_df,
    x='perturb_std',
    y='interactions_to_0.8',
    hue='method',
    errorbar=('ci', 95),
    dodge=0.25,
    ax=axes[1]
)
axes[1].set_title('Interactions to 80% success')
axes[1].set_xlabel('Transition noise std')
axes[1].set_ylabel('Cumulative env interactions')
axes[1].grid(True, alpha=0.3)

if axes[1].get_legend() is not None:
    axes[1].get_legend().remove()
plt.tight_layout()
plt.show()

# 3) AUC(success) comparison
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=results_df, x='perturb_std', y='auc_success', hue='method', ax=ax)
ax.set_title('AUC of success curve (higher is better)')
ax.set_xlabel('Transition noise std')
ax.set_ylabel('AUC(success vs iteration)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 4) Policy quality plot: eval steps (lower is better)
fig, axes = plt.subplots(1, len(PERTURB_LEVELS), figsize=(4 * len(PERTURB_LEVELS), 4), sharey=True)
if len(PERTURB_LEVELS) == 1:
    axes = [axes]

for i, perturb_std in enumerate(PERTURB_LEVELS):
    ax = axes[i]
    subset = steps_curve_df[steps_curve_df['perturb_std'] == perturb_std]
    sns.lineplot(
        data=subset,
        x='iteration',
        y='eval_steps',
        hue='method',
        estimator='mean',
        errorbar=('ci', 95),
        ax=ax
    )
    ax.set_title(f'Eval steps | noise std = {perturb_std:.2f}')
    ax.grid(True, alpha=0.3)
    if i > 0:
        ax.get_legend().remove()

axes[0].set_ylabel('Average steps to goal')
axes[-1].legend(loc='upper right')
plt.tight_layout()
plt.show()

# 5) Diagnostic panel: gradient norm + fitness std
fig, axes = plt.subplots(2, len(PERTURB_LEVELS), figsize=(4 * len(PERTURB_LEVELS), 7), sharex=True)
if len(PERTURB_LEVELS) == 1:
    axes = np.array([[axes[0]], [axes[1]]])

for i, perturb_std in enumerate(PERTURB_LEVELS):
    grad_subset = grad_curve_df[grad_curve_df['perturb_std'] == perturb_std]
    std_subset = fitstd_curve_df[fitstd_curve_df['perturb_std'] == perturb_std]

    sns.lineplot(
        data=grad_subset,
        x='iteration',
        y='gradient_norm',
        hue='method',
        estimator='mean',
        errorbar=('ci', 95),
        ax=axes[0, i]
    )
    axes[0, i].set_title(f'Grad norm | noise std = {perturb_std:.2f}')
    axes[0, i].set_yscale('log')
    axes[0, i].grid(True, alpha=0.3)

    sns.lineplot(
        data=std_subset,
        x='iteration',
        y='fitness_std',
        hue='method',
        estimator='mean',
        errorbar=('ci', 95),
        ax=axes[1, i]
    )
    axes[1, i].set_title(f'Fitness std | noise std = {perturb_std:.2f}')
    axes[1, i].grid(True, alpha=0.3)

    if i > 0:
        if axes[0, i].get_legend() is not None:
            axes[0, i].get_legend().remove()
        if axes[1, i].get_legend() is not None:
            axes[1, i].get_legend().remove()

axes[0, 0].set_ylabel('Gradient norm (log)')
axes[1, 0].set_ylabel('Fitness std')
for i in range(len(PERTURB_LEVELS)):
    axes[1, i].set_xlabel('Iteration')

plt.tight_layout()
plt.show()

C:\Users\jrtam\AppData\Local\Temp\ipykernel_13892\1472355250.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\jrtam\AppData\Local\Temp\ipykernel_13892\1472355250.py:47: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  axes[-1].legend(loc='lower right')
C:\Users\jrtam\AppData\Local\Temp\ipykernel_13892\1472355250.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Result Interpretation

Use the table from the next code cell to summarize which method is preferable by perturbation level.

**Decision rule from plan:**
- LoRA preferred when it reaches thresholds at similar/fewer interactions and keeps similar final success.
- Full ES preferred when perturbation is large and LoRA misses high-success thresholds under the same budget.

After running, report:
1. `time_to_0.8` and `interactions_to_0.8` by perturbation level
2. AUC difference (LoRA - Full)
3. Final success difference (LoRA - Full)

In [ ]:
agg = results_df.groupby(['perturb_std', 'method']).agg(
    final_eval_success=('final_eval_success', 'mean'),
    final_eval_reward=('final_eval_reward', 'mean'),
    final_eval_steps=('final_eval_steps', 'mean'),
    auc_success=('auc_success', 'mean'),
    time_to_0_8=('time_to_0.8', 'mean'),
    interactions_to_0_8=('interactions_to_0.8', 'mean')
).reset_index()

pivot = agg.pivot(index='perturb_std', columns='method')

delta = pd.DataFrame({
    'delta_final_success_lora_minus_full': pivot['final_eval_success']['lora'] - pivot['final_eval_success']['full'],
    'delta_auc_lora_minus_full': pivot['auc_success']['lora'] - pivot['auc_success']['full'],
    'delta_time_to_0_8_lora_minus_full': pivot['time_to_0_8']['lora'] - pivot['time_to_0_8']['full'],
    'delta_interactions_to_0_8_lora_minus_full': pivot['interactions_to_0_8']['lora'] - pivot['interactions_to_0_8']['full']
}).reset_index()

print('Method means by perturbation level:')
display(agg.round(4))

print('LoRA minus Full deltas (negative time/interactions means faster LoRA):')
display(delta.round(4))

print('\nQuick criterion check:')
for _, row in delta.iterrows():
    p = row['perturb_std']
    faster = row['delta_interactions_to_0_8_lora_minus_full'] <= 0
    competitive = row['delta_final_success_lora_minus_full'] >= -0.05
    if faster and competitive:
        verdict = 'LoRA favorable at this perturbation level'
    else:
        verdict = 'Full ES likely favorable at this perturbation level'
    print(f'  noise_std={p:.2f}: {verdict}')

### Export Results to CSV

Run this after the experiment cells to save both run-level and summary tables for reporting.

In [ ]:
from pathlib import Path

if 'results_df' not in globals():
    raise RuntimeError('results_df not found. Run the adaptation experiment cell first.')

if 'agg' not in globals():
    agg = results_df.groupby(['perturb_std', 'method']).agg(
        final_eval_success=('final_eval_success', 'mean'),
        final_eval_reward=('final_eval_reward', 'mean'),
        final_eval_steps=('final_eval_steps', 'mean'),
        auc_success=('auc_success', 'mean'),
        time_to_0_8=('time_to_0.8', 'mean'),
        interactions_to_0_8=('interactions_to_0.8', 'mean')
    ).reset_index()

if 'delta' not in globals():
    pivot = agg.pivot(index='perturb_std', columns='method')
    delta = pd.DataFrame({
        'delta_final_success_lora_minus_full': pivot['final_eval_success']['lora'] - pivot['final_eval_success']['full'],
        'delta_auc_lora_minus_full': pivot['auc_success']['lora'] - pivot['auc_success']['full'],
        'delta_time_to_0_8_lora_minus_full': pivot['time_to_0_8']['lora'] - pivot['time_to_0_8']['full'],
        'delta_interactions_to_0_8_lora_minus_full': pivot['interactions_to_0_8']['lora'] - pivot['interactions_to_0_8']['full']
    }).reset_index()

out_dir = Path('results')
out_dir.mkdir(parents=True, exist_ok=True)

run_path = out_dir / 'es_lora_adaptation_runs.csv'
summary_path = out_dir / 'es_lora_adaptation_summary.csv'
delta_path = out_dir / 'es_lora_adaptation_deltas.csv'

results_df.to_csv(run_path, index=False)
agg.to_csv(summary_path, index=False)
delta.to_csv(delta_path, index=False)

print('Saved CSV files:')
print(f'  {run_path.resolve()}')
print(f'  {summary_path.resolve()}')
print(f'  {delta_path.resolve()}')